## A3 continued: Image preprocessing pipeline

Now that we have leakage-free train / validation / test splits (60/20/20, confirmed zero overlap above), we need to turn the raw PNG files into something a neural network can actually consume: fixed-size, normalised numeric arrays.

**Choices made here, and why:**
- **Resize to 224x224** — this is the standard input size for many pretrained CNN backbones (including the one we use below), so resizing to this size lets us use transfer learning without modifying the network's first layer.
- **Normalize pixel values to the 0-1 range** — raw PNG pixels are 0-255 integers. Neural networks train more stably on small, centred numeric ranges, so we divide by 255.
- **Light augmentation (rotation, zoom, horizontal flip) on training data only** — X-rays can appear at slightly different angles/zoom depending on how the scan was taken. Mild augmentation helps the model generalise rather than memorise exact pixel positions. We do NOT augment validation/test data, since we want those to reflect real, unmodified images the model will see in practice.


In [ ]:
# Add a "filename" column to each split so we can point the image loader
# at the exact file for each row. Images are stored as "<id>.png".
train_split = train_split.copy()
val_split = val_split.copy()
test_split = test_split.copy()

train_split["filename"] = train_split["id"].astype(str) + ".png"
val_split["filename"] = val_split["id"].astype(str) + ".png"
test_split["filename"] = test_split["id"].astype(str) + ".png"

# age_bracket needs to be a plain string column (not pandas' special "category"
# dtype) for Keras's image loader to handle it correctly below.
train_split["age_bracket"] = train_split["age_bracket"].astype(str)
val_split["age_bracket"] = val_split["age_bracket"].astype(str)
test_split["age_bracket"] = test_split["age_bracket"].astype(str)

print(train_split[["id", "filename", "age_bracket"]].head())


In [ ]:
# tensorflow.keras gives us ImageDataGenerator: a tool that reads images from
# disk in small batches (so we never have to load all 12,611 images into memory
# at once), resizes them, normalises them, and optionally augments them.
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)   # target width/height every image gets resized to
BATCH_SIZE = 32         # how many images the model sees per training step

# Training generator: rescales pixels to 0-1, and applies mild augmentation
# only to the training set (never to validation/test).
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,      # randomly rotate up to 10 degrees
    zoom_range=0.1,         # randomly zoom in/out up to 10%
    horizontal_flip=False,  # NOT used: hand X-ray orientation (which side) can
                             # matter medically, so we don't want to flip left/right
)

# Validation/test generators: rescale only, no augmentation — these must reflect
# real, unmodified images so evaluation results are trustworthy.
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)


In [ ]:
# flow_from_dataframe reads images listed in a pandas DataFrame directly from
# a folder on disk, matching each row's "filename" to the actual image file,
# and its "age_bracket" to the label — resizing and batching automatically.

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_split,
    directory=train_images_path,   # the training images folder we found earlier
    x_col="filename",
    y_col="age_bracket",
    target_size=IMG_SIZE,
    class_mode="categorical",      # one-hot encoded labels, for multi-class classification
    batch_size=BATCH_SIZE,
    shuffle=True,                  # shuffle training data each epoch, to avoid the
                                    # model learning any accidental ordering
    seed=42,
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_split,
    directory=train_images_path,   # validation images come from the same folder —
                                    # remember, we carved val/test out of the training set ourselves
    x_col="filename",
    y_col="age_bracket",
    target_size=IMG_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False,   # no need to shuffle — we're not training on this data
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_split,
    directory=train_images_path,
    x_col="filename",
    y_col="age_bracket",
    target_size=IMG_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# class_indices shows us which numeric index Keras assigned to each bracket label —
# we'll need this mapping later when interpreting predictions
print("Class index mapping:", train_generator.class_indices)


## A4: Model Architecture & Justification

**Architecture choice: transfer learning with MobileNetV2 as a frozen backbone, plus a custom classification head.**

**Why transfer learning rather than a custom CNN built from scratch:**
- We have ~7,566 training images spread across 4 classes — enough to fine-tune a classification head, but relatively small for training a deep CNN's convolutional filters entirely from scratch (which typically needs tens/hundreds of thousands of images to learn good low-level filters like edges and textures without overfitting).
- MobileNetV2 was pretrained on ImageNet (1.4 million general images). Its early layers already recognise general shapes, edges, and textures — useful even for X-rays, which are still images with edges and structures, just not photos of everyday objects. We reuse that general visual knowledge and only train new layers on top for our specific bone-age-bracket task.
- MobileNetV2 is lightweight compared to larger backbones (e.g. ResNet50, VGG16), which matters for training speed on a laptop without a dedicated GPU.

**What we rejected and why:** A custom CNN trained from scratch was considered, but was rejected due to the relatively small dataset size for this class count, which risks overfitting without the benefit of pretrained low-level features.

**Input shape:** (224, 224, 3) — 224x224 pixels, 3 colour channels. Note: X-rays are technically grayscale, but MobileNetV2 expects 3-channel input since it was trained on colour images; flow_from_dataframe automatically converts grayscale PNGs to 3-channel by default.

**Output shape:** (4,) — a probability distribution over our 4 age brackets (via softmax), e.g. [0.05, 0.10, 0.70, 0.15] meaning the model is 70% confident the image falls in the third bracket.


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

# Load MobileNetV2 pretrained on ImageNet, without its original classification
# head (include_top=False), since we're replacing that head with our own.
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
)

# Freeze the base model's weights — this means training will NOT change
# MobileNetV2's existing learned filters, only the new layers we add below.
# This keeps training fast and avoids overfitting the pretrained weights
# on our comparatively small dataset.
base_model.trainable = False

# Build the custom classification head on top of the frozen base:
x = base_model.output
x = GlobalAveragePooling2D()(x)   # collapses each feature map to a single average value,
                                    # drastically reducing the number of parameters
                                    # compared to flattening directly
x = Dense(128, activation="relu")(x)   # a fully-connected layer to learn task-specific
                                         # combinations of the pretrained features
x = Dropout(0.3)(x)   # randomly disables 30% of neurons during training, to
                        # reduce overfitting (this is our regularisation choice — see A5)
predictions = Dense(4, activation="softmax")(x)   # 4 = number of age brackets;
                                                     # softmax outputs a probability per class

model = Model(inputs=base_model.input, outputs=predictions)

model.summary()


## A5: Training Process

**Loss function:** categorical crossentropy — the standard loss for multi-class classification where each image belongs to exactly one class.

**Optimizer:** Adam, with a small learning rate (0.0001) — Adam adapts its learning rate per parameter automatically and is a reliable default; we use a smaller-than-default learning rate since we're fine-tuning on top of pretrained weights, where large updates could destabilise the useful features already learned.

**Batch size:** 32 — a common default that balances training speed and memory usage.

**Epochs:** up to 20, with early stopping (see below) — rather than a fixed low number, we let the model train longer but automatically stop if it stops improving, to avoid both under- and over-training.

**Overfitting mitigation:** Dropout (0.3, already added above), plus **early stopping** (below), which halts training if validation loss stops improving for several epochs and restores the best-performing weights — directly addressing the risk of the model memorising the training set rather than generalising.


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# Stop training early if validation loss doesn't improve for 3 consecutive epochs,
# and roll back to the best-performing weights seen during training — this is
# our main defence against overfitting on top of Dropout.
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

EPOCHS = 20

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stop],
)


In [ ]:
# Plot training vs validation loss and accuracy across epochs — this is the
# evidence the brief asks for in A5 to show whether the model overfit.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="Training loss")
axes[0].plot(history.history["val_loss"], label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training vs Validation Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Training accuracy")
axes[1].plot(history.history["val_accuracy"], label="Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training vs Validation Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

# WRITE A SHORT COMMENT HERE (in a markdown cell below) once you see the plot:
# - If training and validation lines stay close together, the model is generalising well.
# - If training loss keeps dropping while validation loss rises, that's overfitting —
#   even with dropout and early stopping, note this honestly in your write-up if it happens.


**[Fill in after running the cell above]:** Describe what the loss/accuracy curves actually show for your run — do training and validation track closely, or diverge? This written comment is required by the brief (A5) and must reflect your real output, not a generic statement.


## A6: Evaluation & Results

We evaluate on the **test set** (2,523 images) — data the model has never seen during training or validation-based early stopping decisions. This is important: evaluating on validation data here would be misleading, since validation data already influenced which weights we kept (via early stopping).


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Run predictions on every test image. The generator yields batches in a fixed
# order (shuffle=False was set above), so predictions line up with test_split rows.
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)

# Convert predicted probabilities into a single predicted class index per image
predicted_classes = np.argmax(predictions, axis=1)

# True class indices, as assigned by the generator
true_classes = test_generator.classes

# Map numeric indices back to readable bracket labels for the report
class_labels = list(test_generator.class_indices.keys())

print(classification_report(true_classes, predicted_classes, target_names=class_labels))


In [ ]:
# A confusion matrix shows exactly which classes get confused with which —
# far more informative than a single accuracy number, and directly addresses
# the brief's warning against reporting only one aggregate metric.
cm = confusion_matrix(true_classes, predicted_classes)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_xlabel("Predicted bracket")
ax.set_ylabel("True bracket")
ax.set_title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()


In [ ]:
# Show at least one CORRECT and one INCORRECT prediction with images,
# as the brief explicitly requires in A6 — including a mistake, not just wins.
import matplotlib.image as mpimg
import os

test_split_reset = test_split.reset_index(drop=True)

correct_idx = None
incorrect_idx = None
for i in range(len(predicted_classes)):
    if predicted_classes[i] == true_classes[i] and correct_idx is None:
        correct_idx = i
    if predicted_classes[i] != true_classes[i] and incorrect_idx is None:
        incorrect_idx = i
    if correct_idx is not None and incorrect_idx is not None:
        break

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, idx, title in [(axes[0], correct_idx, "Correct prediction"),
                        (axes[1], incorrect_idx, "Incorrect prediction")]:
    img_path = os.path.join(train_images_path, test_split_reset.loc[idx, "filename"])
    img = mpimg.imread(img_path)
    ax.imshow(img, cmap="gray")
    true_label = class_labels[true_classes[idx]]
    pred_label = class_labels[predicted_classes[idx]]
    confidence = predictions[idx][predicted_classes[idx]] * 100
    ax.set_title(f"{title}\nTrue: {true_label}\nPredicted: {pred_label} ({confidence:.1f}%)")
    ax.axis("off")

plt.tight_layout()
plt.show()


**[Fill in after reviewing the confusion matrix]:** Write 2-3 honest sentences on where the model struggles most (e.g. which adjacent brackets get confused most often — this is common near bracket boundaries, since a bone age of 97 months and 95 months look nearly identical but fall in different brackets), and your best guess as to why, based on what you see in the confusion matrix above.


## A7: Model Export

Saving the trained model so the Streamlit portal (Part B) can load it later, and documenting the exact preprocessing that must be replicated at inference time.


In [ ]:
# Save the full trained model (architecture + weights + optimizer state) in
# Keras's native format. Record the exact library version too, since model
# file compatibility can break across major library version changes.
import tensorflow as tf

model.save("bone_age_classifier.keras")
print("TensorFlow version used to save this model:", tf.__version__)


**Preprocessing pipeline that MUST be exactly replicated in the Streamlit app (Part B):**

1. Load the uploaded image.
2. Resize to **224x224** pixels.
3. Ensure **3 channels** (RGB) — if the uploaded image is grayscale, convert it to 3-channel.
4. **Rescale pixel values by dividing by 255** (to bring them into the 0-1 range) — no other normalisation (e.g. no mean subtraction) was used.
5. Add a batch dimension (shape becomes (1, 224, 224, 3)) before passing to `model.predict()`.
6. The model's class index order matches `train_generator.class_indices` printed above — the app must use that exact same order when mapping predicted index back to a bracket label. Copy that dictionary's output into the app.

Any deviation from these exact steps (e.g. forgetting to rescale, or resizing differently) will cause the app to produce wrong or nonsensical predictions, even though the model itself is unchanged — this is the single most common reason a working notebook model "breaks" once wired into an app, per the brief's own warning.


## A8: Limitations, Ethics & Next Steps

**Data bias and generalisation limits:**
- The RSNA Bone Age dataset was collected from a specific set of US hospitals (Stanford, Colorado, UCLA), so the model may not generalise well to populations with different demographics, nutrition patterns, or ethnic backgrounds not well represented in this data, all of which can affect skeletal maturation.
- All images come from a limited set of scanning equipment/protocols; X-rays from different machines, exposure settings, or non-US clinical practices may look different enough to reduce model accuracy.
- Converting a continuous age (months) into 4 discrete brackets means the model's usefulness is bounded by those bracket widths — it cannot express fine-grained age estimates, and predictions near bracket boundaries are the least reliable (as seen in the confusion matrix in A6).

**Ethical considerations:**
- **False negatives** here (predicting a younger/older bracket than reality) could lead to a missed or delayed referral for a genuine growth or endocrine concern.
- **False positives** could cause unnecessary parental anxiety or further unnecessary clinical investigation.
- Because consequences run in both directions and this model was trained on a relatively narrow, dataset-specific population, it must never be used as a standalone diagnostic decision-maker. It requires oversight from a qualified radiologist or paediatrician who can weigh the prediction against the full clinical picture (patient history, physical exam, other tests).

**What this project cannot claim:** This is a coursework prototype demonstrating an ML pipeline, not a validated clinical tool. It has not been tested against a clinically diverse population, has not undergone any regulatory review, and its accuracy figures reflect performance on a single dataset's own held-out split, not real-world deployment conditions.

**What would be needed before real clinical use:** Validation on independent, demographically diverse datasets; review and sign-off from clinical experts; formal regulatory approval processes (e.g. FDA clearance in the US); ongoing monitoring for performance drift once deployed; and clear guardrails ensuring it's used only as a decision-support aid alongside, not instead of, a qualified clinician.
